<a href="https://colab.research.google.com/github/vitoriaferreirap/DeepLearning/blob/main/CNN_Computer_Vision/04_DCL_Treinamento_Predicao_de_Anotacoes_Frames.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Objetivo desta etapa:
- Treinar a arquitetura ResNet-50 para reconhecer pontos anatômicos (cascos/articulações) nos frames selecionados. O resultado esperado é uma rede capaz de automatizar a anotação dos frames restantes do vídeo. Analisar a aplicação de hiperparametros diferentes.

## First, go to "Runtime" ->"change runtime type"->select "Python3", and then select "GPU"

As the COLAB environments were updated to CUDA 12.X and Python 3.11, we need to install DeepLabCut and TensorFlow in a distinct way to get TensorFlow to connect to the GPU.

In [2]:
# this will take a couple of minutes to install all the dependencies!
!pip install --pre deeplabcut

  Using cached deeplabcut-3.0.0rc13-py3-none-any.whl.metadata (32 kB)
  Using cached albumentations-1.4.3-py3-none-any.whl.metadata (37 kB)
  Using cached dlclibrary-0.0.11-py3-none-any.whl.metadata (4.2 kB)
  Using cached filterpy-1.4.5-py3-none-any.whl
  Using cached ruamel_yaml-0.19.1-py3-none-any.whl.metadata (16 kB)
  Using cached imgaug-0.4.0-py2.py3-none-any.whl.metadata (1.8 kB)
  Using cached matplotlib-3.8.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.8 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
  Using cached opencv_python_headless-4.13.0.90-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached opencv_python_headless-4.12.0.88-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (19 kB)
  Using cached 

**(Be sure to click "RESTART RUNTIME" if it is displayed above before moving on !)** You will see this button at the output of the cells above ^.

In [1]:
import deeplabcut

Loading DLC 3.0.0rc13...
DLC loaded in light mode; you cannot use any GUI (labeling, relabeling and standalone GUI)


## Link your Google Drive (with your labeled data, or the demo data):

### First, place your project folder into you google drive! "i.e. move the folder named "Project-YourName-TheDate" into google drive.

In [2]:
# Now, let's link to your GoogleDrive. Run this cell and follow the authorization instructions:
# (We recommend putting a copy of the github repo in your google drive if you are using the demo "examples")

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


YOU WILL NEED TO EDIT THE PROJECT PATH **in the config.yaml file** TO BE SET TO YOUR GOOGLE DRIVE LINK!

Typically, this will be: `/content/drive/My Drive/yourProjectFolderName`


In [3]:
# PLEASE EDIT THIS:
project_folder_name = "deeplabcut/30FPS-cavaloTeste1-2026-03-03"
video_type = "mp4" #, mp4, MOV, or avi, whatever you uploaded!

# No need to edit this, we are going to assume you put videos you want to analyze
# in the "videos" folder, but if this is NOT true, edit below:
videofile_path = f"/content/drive/MyDrive/{project_folder_name}/videos/teste1_30fps.mp4"
print(videofile_path)

# The prediction files and labeled videos will be saved in this `labeled-videos` folder
# in your project folder; if you want them elsewhere, you can edit this;
# if you want the output files in the same folder as the videos, set this to an empty string.
destfolder = f"/content/drive/MyDrive/{project_folder_name}/labeled-videos"

#No need to edit this, as you set it when you passed the ProjectFolderName (above):
path_config_file = f"/content/drive/MyDrive/{project_folder_name}/config.yaml"
print(path_config_file)

# This creates a path variable that links to your Google Drive project

/content/drive/MyDrive/deeplabcut/30FPS-cavaloTeste1-2026-03-03/videos/teste1_30fps.mp4
/content/drive/MyDrive/deeplabcut/30FPS-cavaloTeste1-2026-03-03/config.yaml


## Create a training dataset:

### You must do this step inside of Colab

After running this script the training dataset is created and saved in the project directory under the subdirectory **'training-datasets'**

This function also creates new subdirectories under **dlc-models-pytorch** and appends the project config.yaml file with the correct path to the training and testing pose configuration file. These files hold the parameters for training the network. Such an example file is provided with the toolbox and named as **pytorch_config.yaml**.

Now it is the time to start training the network!

In [4]:
# There are many more functions you can set here, including which network to use!
# Check the docstring for `create_training_dataset` for all options you can use!

deeplabcut.create_training_dataset(
  path_config_file, net_type="resnet_50", engine=deeplabcut.Engine.PYTORCH
)

[(0.95,
  2,
  (array([26, 35, 59, 28, 11,  2, 34, 58, 40, 22,  4, 10, 30, 41, 33, 43, 49,
           7, 14, 32, 50, 29, 42, 54, 18, 56, 27, 15,  5, 31, 16, 51, 20, 52,
           8, 13, 25, 37, 17, 45, 48, 57, 38,  1, 12, 46, 24,  6, 23, 36, 21,
          19,  9, 39, 55,  3,  0]),
   array([53, 47, 44])))]

## Start training:
This function trains the network for a specific shuffle of the training dataset.

In [6]:
# Let's also change the display and save_epochs just in case Colab takes away
# the GPU... If that happens, you can reload from a saved point using the
# `snapshot_path` argument to `deeplabcut.train_network`:
#   deeplabcut.train_network(..., snapshot_path="/content/.../snapshot-050.pt")

# Typically, you want to train to ~200 epochs. We set the batch size to 8 to
# utilize the GPU's capabilities.

# More info and there are more things you can set:
#   https://deeplabcut.github.io/DeepLabCut/docs/standardDeepLabCut_UserGuide.html#g-train-the-network

deeplabcut.train_network(
    path_config_file,
    shuffle=1,
    displayiters=100,
    saveiters=1000,
    maxiters=15000,
    batch_size=1
)

# This will run until you stop it (CTRL+C), or hit "STOP" icon, or when it hits the end.

Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [0.5, 1.25]
      translation: 0
    crop_sampling:
      width: 448
      height: 448
      max_shift: 0.1
      method: hybrid
    gaussian_noise: 12.75
    motion_blur: True
    normalize_images: True
device: auto
inference:
  multithreading:
    enabled: True
    queue_length: 4
    timeout: 30.0
  compile:
    enabled: False
    backend: inductor
  autocast:
    enabled: False
metadata:
  project_path: /content/drive/MyDrive/deeplabcut/30FPS-cavaloTeste1-2026-03-03
  pose_config_path: /content/drive/MyDrive/deeplabcut/30FPS-cavaloTeste1-2026-03-03/dlc-models-pytorch/iteration-0/30FPSMar3-trainset95shuffle1/train/pytorch_config.yaml
  bodyparts: ['Ext_Joelho', 'Ext_CanelaDianteira', 'Ext_CascoDianteiro', 'Ext_Perna', 'Ext_Jarrete', 'Ext_CanelaTraseira', 'Ext_CascoTraseiro', 'Int_Joelho', 'Int_CanelaDianteir

Note, that **when you hit "STOP" you will get a `KeyboardInterrupt` "error"! No worries! :)**

## Start evaluating:
This function evaluates a trained model for a specific shuffle/shuffles at a particular state or all the states on the data set (images)
and stores the results as .csv file in a subdirectory under **evaluation-results-pytorch**

In [7]:
deeplabcut.evaluate_network(path_config_file, plotting=True)
# Here you want to see a low pixel error! Of course, it can only be as
# good as the labeler, so be sure your labels are good!


Evaluation scorer: DLC_Resnet50_30FPSMar3shuffle1_snapshot_best-150


100%|██████████| 3/3 [00:00<00:00, 11.41it/s]


Evaluation results file: DLC_Resnet50_30FPSMar3shuffle1_snapshot_best-150-results.csv
Evaluation results for DLC_Resnet50_30FPSMar3shuffle1_snapshot_best-150-results.csv (pcutoff: 0.1):
train rmse            167.09
train rmse_pcutoff    165.80
train mAP              10.31
train mAR              17.02
test rmse              74.80
test rmse_pcutoff      74.80
test mAP               33.37
test mAR               33.33
Name: (0.95, 1, 150, -1, 0.1), dtype: float64


## There is an optional refinement step you can do outside of Colab:
- if your pixel errors are not low enough, please check out the protocol guide on how to refine your network!
- You will need to adjust the labels **outside of Colab!** We recommend coming back to train and analyze videos...
- Please see the repo and protocol instructions on how to refine your data!

## Start Analyzing videos:
This function analyzes the new video. The user can choose the best model from the evaluation results and specify the correct snapshot index for the variable **snapshotindex** in the **config.yaml** file. Otherwise, by default the most recent snapshot is used to analyse the video.

The results are stored in hd5 file in the same directory where the video resides.

In [8]:
deeplabcut.analyze_videos(
    path_config_file,
    videofile_path,
    videotype=video_type,
    destfolder=destfolder,
)

Creating the output folder /content/drive/MyDrive/deeplabcut/30FPS-cavaloTeste1-2026-03-03/labeled-videos
Analyzing videos with /content/drive/MyDrive/deeplabcut/30FPS-cavaloTeste1-2026-03-03/dlc-models-pytorch/iteration-0/30FPSMar3-trainset95shuffle1/train/snapshot-best-150.pt
Using scorer: DLC_Resnet50_30FPSMar3shuffle1_snapshot_best-150
Starting to analyze /content/drive/MyDrive/deeplabcut/30FPS-cavaloTeste1-2026-03-03/videos/teste1_30fps.mp4
Video metadata: 
  Overall # of frames:    464
  Duration of video [s]:  15.47
  fps:                    30.0
  resolution:             w=1080, h=1920

Running pose prediction with batch size 8


100%|██████████| 464/464 [00:26<00:00, 17.51it/s]


Saving results in /content/drive/MyDrive/deeplabcut/30FPS-cavaloTeste1-2026-03-03/labeled-videos/teste1_30fpsDLC_Resnet50_30FPSMar3shuffle1_snapshot_best-150.h5 and /content/drive/MyDrive/deeplabcut/30FPS-cavaloTeste1-2026-03-03/labeled-videos/teste1_30fpsDLC_Resnet50_30FPSMar3shuffle1_snapshot_best-150_full.pickle
The videos are analyzed. Now your research can truly start!
You can create labeled videos with 'create_labeled_video'.
If the tracking is not satisfactory for some videos, consider expanding the training set. You can use the function 'extract_outlier_frames' to extract a few representative outlier frames.



'DLC_Resnet50_30FPSMar3shuffle1_snapshot_best-150'

## Plot the trajectories of the analyzed videos:
This function plots the trajectories of all the body parts across the entire video. Each body part is identified by a unique color.

In [ ]:
deeplabcut.plot_trajectories(
    path_config_file,
    videofile_path,
    videotype=video_type,
    destfolder=destfolder,
)

Loading  /content/drive/MyDrive/deeplabcut/30FPS-cavaloTeste1-2026-03-03/videos/teste1_30fps.mp4 and data.
Plots created! Please check the directory "plot-poses" within the video directory


Now you can look at the plot-poses file and check the "plot-likelihood.png" might want to change the "p-cutoff" in the config.yaml file so that you have only high confidnece points plotted in the video. i.e. ~0.8 or 0.9. The current default is 0.4.

## Create labeled video:
This function is for visualization purpose and can be used to create a video in .mp4 format with labels predicted by the network. This video is saved in the same directory where the original video resides.

In [ ]:
deeplabcut.create_labeled_video(
    path_config_file,
    videofile_path,
    videotype=video_type,
    destfolder=destfolder,
)

Starting to process video: /content/drive/MyDrive/deeplabcut/30FPS-cavaloTeste1-2026-03-03/videos/teste1_30fps.mp4
Loading /content/drive/MyDrive/deeplabcut/30FPS-cavaloTeste1-2026-03-03/videos/teste1_30fps.mp4 and data.
Duration of video [s]: 15.47, recorded with 30.0 fps!
Overall # of frames: 464 with cropped frame dimensions: 1080 1920
Generating frames and creating video.


100%|██████████| 464/464 [00:11<00:00, 41.33it/s]


[True]